## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [45]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0\
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30 \
              langchain-huggingface

!pip install -q \
sentence-transformers==3.4.1 \
transformers==4.49.0 \
huggingface_hub==0.28.1

!pip install -q pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.3.1 requires huggingface-hub>=0.33.4, but you have huggingface-hub 0.28.1 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.28.1 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.28.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.1 MB/s eta 0:00:00


> **Objective:** Install pinned versions of LangChain, Chroma, PyMuPDF, tiktoken, sentence-transformers, transformers and the Hugging Face client.
>
> **Interpretation:** Pinned versions make the environment reproducible. Dependency-conflict warnings are expected in Colab and can be ignored, but the runtime must be restarted afterwards.

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
# Install llama-cpp-python with GPU support for running LLaMA models
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 --force-reinstall --upgrade --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 154.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 296.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 394.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 219.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 300.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.3.1 requires huggingface-hub>=0.33.4, but you have huggingface-hub 0.28.1 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.28.1 which is incompatible.
gradio 6.

> **Objective:** Compile `llama-cpp-python` with CUDA (cuBLAS) support so the LLaMA model can run on the GPU.
>
> **Interpretation:** Without GPU support a 13B model would be far too slow. This step needs a GPU runtime and can take several minutes to build.

In [ ]:
# Install Hugging Face Hub client library for downloading models
!pip install huggingface_hub==0.20.3 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.49.0 requires huggingface-hub<1.0,>=0.26.0, but you have huggingface-hub 0.20.3 which is incompatible.
langchain-huggingface 0.3.1 requires huggingface-hub>=0.33.4, but you have huggingface-hub 0.20.3 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.20.3 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.20.3 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.20.3 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.20.3 which is incompatible.
accelerate 1.14.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.20.3 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

## **LLM with Prompt Engineering Response**

#### **Download LLaMA-2 13B Chat Model**

In [ ]:
from huggingface_hub import hf_hub_download

model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"
model_path = hf_hub_download(                     # Download the model file from Hugging Face Hub and return its local path
    repo_id=model_name_or_path,
    filename=model_basename
)

> **Objective:** Download the quantized `Llama-2-13B-chat` model (GGUF, Q5_K_M) from the Hugging Face Hub and return its local path.
>
> **Interpretation:** Q5_K_M is 5-bit quantization: it keeps most of the model quality while cutting memory use to a size that fits on a single Colab-class GPU. The model runs locally, so no paid API is needed.

#### **Initialize LLaMA Model with Configuration**

In [ ]:
from llama_cpp import Llama

lcpp_llm = Llama(
    model_path=model_path, # Path to the downloaded GGUF model
    n_threads=4,            # Number of CPU threads to use
    n_batch=512,            # Batch size for prompt processing
    n_gpu_layers=-1,        # Number of layers to offload to GPU (-1 for all)
    n_ctx=4096              # Context window
)

llama_model_loader: loaded meta data with 19 key-value pairs and 363 tensors from /root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q5_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 5120
llama_model_loader: - kv   4:                          llama.block_count u32              = 40
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 13824
llama_model_loader: - kv   6:                 llama.rope.dimension_

> **Objective:** Load the model with a 4,096-token context window, all layers offloaded to the GPU (`n_gpu_layers=-1`), 4 CPU threads and batch size 512.
>
> **Interpretation:** The load log confirms a LLaMA architecture with 40 layers and a 4,096-token context. That 4,096-token limit covers prompt + answer together, which becomes the main constraint in the RAG section.

In [ ]:
# Baseline system prompt: defines the role, tone and answer rules for the LLM
# (no retrieved context is used in this section).
system_prompt = """
You are a medical expert AI assistant that supports healthcare professionals.

Rules:
- Give concise, accurate, evidence-based answers.
- Use short sentences or a numbered list. Do not repeat the question.
- If you are not certain, say so instead of guessing.
- Do not invent references, guideline names or page numbers.
- Do not give personal medical advice. Keep a professional, neutral tone.

Example:
Question: What are the common symptoms of appendicitis?
Answer: Abdominal pain (often starting near the navel and shifting to the lower right abdomen), nausea, vomiting, loss of appetite and low-grade fever.
"""

> **Objective:** Define the baseline (no retrieval) system prompt: role, answer rules and one worked example.
>
> **Interpretation:** This is prompt engineering only. The model can rely on its own memory, so nothing it says can be checked against a source.

#### **Response Function**

In [ ]:
# function to generate, process, and return the response from the LLM
def prompt_engineering_response(user_prompt):
    # Put the system message first, then the user question, then anchor with "Answer:"
    prompt = f"""{system_prompt}

Question: {user_prompt}

Answer:"""

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=500,       # Max number of tokens to generate
        temperature=0.3,      # Sampling temperature
        top_p=0.95,           # Top-p sampling
        stop=["Q:", "\n"],    # Stop generating when "Q:" or a new line is encountered
        echo=False            # Do not echo the prompt in the output
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"].strip()
    return response_text

> **Objective:** Wrap the prompt, call the model (temperature 0.3, top-p 0.95, max 500 tokens) and return the cleaned text.
>
> **Interpretation:** A low temperature gives steady, factual wording. Caution: the stop list contains `"\n"`, so generation ends at the first line break and multi-line answers are cut to one line.

## Question Answering using LLM with Prompt Engineering

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"

llm_response_1 = prompt_engineering_response(question_1)
print(llm_response_1)


llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      23.79 ms /    41 runs   (    0.58 ms per token,  1723.70 tokens per second)
llama_print_timings: prompt eval time =     719.75 ms /    48 tokens (   14.99 ms per token,    66.69 tokens per second)
llama_print_timings:        eval time =    2931.92 ms /    40 runs   (   73.30 ms per token,    13.64 tokens per second)
llama_print_timings:       total time =    3807.86 ms /    88 tokens


The Surviving Sepsis Campaign (SSC) guidelines provide a comprehensive protocol for managing sepsis in a critical care unit. Here are the key recommendations:


> **Objective:** Ask the baseline model for the sepsis management protocol in critical care.
>
> **Interpretation:** The answer names the Surviving Sepsis Campaign but ends with "Here are the key recommendations:" and lists none. The model knows the right guideline but the protocol steps are missing.

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

llm_response_2 = prompt_engineering_response(question_2)
print(llm_response_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      17.95 ms /    24 runs   (    0.75 ms per token,  1336.82 tokens per second)
llama_print_timings: prompt eval time =     417.09 ms /    37 tokens (   11.27 ms per token,    88.71 tokens per second)
llama_print_timings:        eval time =    1701.95 ms /    23 runs   (   74.00 ms per token,    13.51 tokens per second)
llama_print_timings:       total time =    2255.06 ms /    60 tokens


Appendicitis is a medical emergency that requires prompt treatment. The common symptoms of appendicitis include:


> **Objective:** Test a three-part question: symptoms of appendicitis, whether medicine can cure it, and the surgical procedure.
>
> **Interpretation:** Only the opening sentence ("a medical emergency that requires prompt treatment") is returned. None of the three parts is actually answered.

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

llm_response_3 = prompt_engineering_response(question_3)
print(llm_response_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      22.74 ms /    35 runs   (    0.65 ms per token,  1539.34 tokens per second)
llama_print_timings: prompt eval time =     423.97 ms /    41 tokens (   10.34 ms per token,    96.70 tokens per second)
llama_print_timings:        eval time =    2578.69 ms /    34 runs   (   75.84 ms per token,    13.18 tokens per second)
llama_print_timings:       total time =    3162.66 ms /    75 tokens


Sudden patchy hair loss, also known as alopecia areata, can have various causes and effective treatments. Here are some possible causes and treatment options:


> **Objective:** Test a cause-and-treatment question about sudden patchy hair loss.
>
> **Interpretation:** The model correctly names alopecia areata, then stops before listing any cause or treatment.

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

llm_response_4 = prompt_engineering_response(question_4)
print(llm_response_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      37.36 ms /    66 runs   (    0.57 ms per token,  1766.69 tokens per second)
llama_print_timings: prompt eval time =     418.17 ms /    35 tokens (   11.95 ms per token,    83.70 tokens per second)
llama_print_timings:        eval time =    5026.91 ms /    65 runs   (   77.34 ms per token,    12.93 tokens per second)
llama_print_timings:       total time =    5681.97 ms /   100 tokens


Treatment options for a person who has sustained a physical injury to brain tissue and experienced temporary or permanent impairment of brain function will depend on the severity and location of the injury, as well as the individual's overall health and medical history. Here are some common treatment approaches that may be recommended:


> **Objective:** Test a neurology question about treatment after brain-tissue injury.
>
> **Interpretation:** The answer says treatment depends on severity, location and overall health, then stops. It is generic and contains no specific treatment.

#### **Create and Display Results DataFrame**

In [ ]:
# Create the DataFrame
import pandas as pd

prompt_result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "prompt_Engineering_responses": [llm_response_1, llm_response_2, llm_response_3, llm_response_4]})

# Display the DataFrame
prompt_result_df.head()

,questions,prompt_Engineering_responses
0,What is the protocol for managing sepsis in a ...,The Surviving Sepsis Campaign (SSC) guidelines...
1,"What are the common symptoms for appendicitis,...",Appendicitis is a medical emergency that requi...
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, also known as alopeci..."
3,What treatments are recommended for a person w...,Treatment options for a person who has sustain...


> **Objective:** Collect the four baseline answers in a DataFrame for side-by-side comparison with RAG.
>
> **Interpretation:** The table makes the truncation visible: every response is a single short sentence.

#### **Observations:**

* **Right topic, no substance.** The baseline recognises the subject in every case (Surviving Sepsis Campaign for Q1, alopecia areata for Q3), so the base LLaMA-2 model has general medical knowledge.
* **All four answers stop after one opening sentence** (for example, "Here are the key recommendations:" with no list after it). The cause is `stop=["Q:", "\n"]` in the response function: generation ends at the first line break, so numbered lists never appear.
* **No verifiable source.** Nothing in the answer can be traced back to a manual or guideline page, which is a risk in a clinical setting.
* **Fast.** Each answer took about 2-6 seconds (roughly 13 tokens/second), because the prompt is tiny and there is no retrieval step.

## **RAG Response**

## **Data Preparation for RAG**

### **Loading the data**

In [ ]:
# Mount Google Drive to the /content/drive directory to access the files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
from langchain_community.document_loaders import PyMuPDFLoader

# Load Merck Manual PDF
# NOTE: update this path to match where you uploaded the PDF in your own Google Drive
pdf_path = "/content/drive/MyDrive/Dataset/GenAIDataset/medical_diagnosis_manual.pdf"
loader = PyMuPDFLoader(pdf_path)

# Load as LangChain Documents
document = loader.load()

print(f"Total pages loaded: {len(document)}")

Total pages loaded: 4114


> **Objective:** Load the Merck Manual PDF page by page as LangChain documents using PyMuPDF.
>
> **Interpretation:** 4,114 pages were loaded, so the whole manual was read. Each page keeps metadata (source file and page number) that can be used for page-level citations.

### Data Overview

Display the content of page number 16 and 17

In [32]:
for i in range(15, 17):
    print("=" * 80)
    print(f"Page {i + 1}")
    print("=" * 80)
    print(document[i].page_content)
    print("\n")

Page 16
degree. The book received critical acclaim and sold over 2 million copies. The Second Home Edition was
released in 2003. Merck's commitment to providing comprehensive, understandable medical information
to all people continued with The Merck Manual Home Health Handbook, published in 2009.
The Merck Manual of Health & Aging , published in 2004, continued Merck's commitment to education
and geriatric care, providing information on aging and the care of older people in words understandable
by the lay public.
In 2008, The Merck Manual of Patient Symptoms  was introduced to complement The Merck Manual
and was intended to help newcomers to clinical diagnosis approach patients who present with certain
common symptoms.
As part of its commitment to ensuring that all who need and want medical information can get it, Merck
provides the content of these Merck Manuals on the web for free (www.merckmanuals.com). Registration
is not required, and use is unlimited. The web publications are con

> **Objective:** Print pages 16 and 17 as a quick check that text was extracted correctly.
>
> **Interpretation:** The text is clean, readable prose, so OCR is not needed. Page 16 is front matter about the history of the Merck publications, showing that the PDF also contains non-clinical pages that end up in the index.

## **Data Chunking**

Split the document into Chunks and display the total chunks

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=512,      # Maximum tokens per chunk
    chunk_overlap=50     # Overlap between consecutive chunks to preserve context
)

# Split the loaded documents into smaller overlapping chunks
docs = text_splitter.split_documents(document)

print(f"Total chunks: {len(docs)}")

Total chunks: 8797


> **Objective:** Split the pages into overlapping chunks of 512 tokens (50-token overlap) using a tiktoken-based recursive splitter.
>
> **Interpretation:** The 4,114 pages became 8,797 chunks (about 2 per page). 512 tokens is large enough to hold a full clinical idea, and the overlap prevents important sentences from being cut at chunk borders.

### Embedding

Generate vector embeddings for the text chunks using the open-source Hugging Face model `BAAI/bge-base-en-v1.5` (normalized embeddings).

In [34]:
from langchain_huggingface import HuggingFaceEmbeddings

import warnings
warnings.filterwarnings('ignore')

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True}  # Converts embeddings to unit vectors
)

> **Objective:** Load the `BAAI/bge-base-en-v1.5` embedding model with normalized vectors.
>
> **Interpretation:** Normalized embeddings make similarity search behave like cosine similarity. BGE-base is a strong open-source model, so embedding costs nothing and runs locally.

In [50]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/drive/MyDrive/Dataset/GenAIDataset/medical_diagnosis_manual.pdf")

documents = loader.load()

> **Objective:** Load the same PDF a second time with `PyPDFLoader` into the variable `documents`.
>
> **Interpretation:** This variable is not used later; the index is built from `docs` (from the first loader). The cell can be removed without changing any result.

In [53]:
from langchain_community.vectorstores import Chroma

# Building the vector store and saving it to disk for future use
persist_directory = "/content/drive/MyDrive/Dataset/GenAIDataset/rag_healthcare_llm"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print(f"Vector database saved to:\n{persist_directory}")

Vector database saved to:
/content/drive/MyDrive/Dataset/GenAIDataset/rag_healthcare_llm


> **Objective:** Embed all 8,797 chunks and store them in a Chroma vector database saved to Google Drive.
>
> **Interpretation:** This is the slowest one-time step. Because the database is persisted, later sessions can reload it instead of re-embedding the whole manual.

### Retriever

Retrieval and Response Generation using Vector Search

In [54]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

> **Objective:** Create a retriever that returns the 5 most similar chunks for a question.
>
> **Interpretation:** k=5 is a balance: more chunks give more evidence but use more of the 4,096-token window (RAG prompts already reached 2,600-3,600 tokens).

In [55]:
# RAG system prompt: restricts the model to the retrieved Merck Manual context
medical_system_message = """
You are a clinical knowledge assistant that supports healthcare professionals by answering questions from trusted medical sources such as the Merck Manuals.

The user message contains retrieved excerpts that start with ###Context, followed by the question that starts with ###Question.

Rules:
1. Answer using ONLY the provided context. Do not add outside knowledge.
2. Be concise, clinically relevant and accurate.
3. Cite the source (document name and page or section) shown in the context.
4. If the context does not contain the answer, reply exactly: "Sorry, this is out of my knowledge base."
5. Do not give personal medical advice. Keep a professional, neutral and safe tone.

Response format:
Answer:
<answer based on the context>

Source:
<document, section, page>
"""

In [56]:
# RAG user prompt template: {context} = retrieved chunks, {question} = user query
medical_user_message_template = """
###Context
{context}

###Question
{question}
"""

> **Objective:** Define the RAG system message and the user template that inserts retrieved context and the question.
>
> **Interpretation:** The rules force the model to use only the context, cite sources and say "Sorry, this is out of my knowledge base." when the manual does not help. This is the main guard against hallucination.

### Response Function

In [57]:
def generate_rag_response(user_input, retriever,
                          system_message, user_message_template,
                          k=5, max_tokens=500, temperature=0.3, top_p=0.95):

    # Retrieve relevant document chunks
    relevant_chunks = retriever.invoke(user_input)
    if not relevant_chunks:
        return "Sorry, this is out of my knowledge base."

    # Combine document chunks with source info
    context_for_query = "\n\n".join([f"Source: {d.metadata.get('source','Unknown')}\n{d.page_content}" for d in relevant_chunks])

    # Fill user message template
    user_message = user_message_template.format(
        context=context_for_query,
        question=user_input
    )

    prompt = f"""{system_message}

{user_message}

Answer:
"""

    # Generate the response
    try:
        response = lcpp_llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            stop=["###Question", "###Context"],
            echo=False
        )
        return response["choices"][0]["text"].strip()

    except Exception as e:
        return f"Sorry, I encountered the following error:\n{e}"

> **Objective:** Retrieve chunks, build the context string, fill the template, call the LLM and return the answer (with a fallback message when nothing is retrieved).
>
> **Interpretation:** Each chunk is labelled only with `metadata['source']`, which is the PDF file path. The page number is available but not passed, so citations cannot be page-specific yet.

## Question Answering using RAG

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [58]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"

# Call the RAG response function
response_with_rag_1 = generate_rag_response(
    user_input=question_1,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

# Print the response
print(response_with_rag_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =     236.02 ms /   367 runs   (    0.64 ms per token,  1554.96 tokens per second)
llama_print_timings: prompt eval time =   10192.37 ms /  3590 tokens (    2.84 ms per token,   352.22 tokens per second)
llama_print_timings:        eval time =   39323.18 ms /   366 runs   (  107.44 ms per token,     9.31 tokens per second)
llama_print_timings:       total time =   51751.07 ms /  3956 tokens


According to the Merck Manual, the protocol for managing sepsis in a critical care unit includes:

1. Early recognition of sepsis with signs of systemic inflammation (e.g., fever, tachycardia, tachypnea) and prompt activation of the sepsis team.
2. Aggressive fluid resuscitation with 0.9% normal saline to restore intravascular volume and improve perfusion.
3. Broad-spectrum antibiotics within the first hour of recognition of sepsis, guided by culture results and sensitivities.
4. Drainage of abscesses and excision of necrotic tissue to eliminate foci of infection.
5. Normalization of blood glucose levels with continuous IV insulin infusion to improve outcome.
6. Close monitoring of vital signs, blood chemistry, and organ function, with frequent assessment of fluid status and oxygenation.
7. Early recognition and management of organ dysfunction, including cardiovascular, respiratory, and renal failure.
8. Surgical intervention as needed for source control or management of complications.

> **Objective:** Answer the sepsis question using retrieved Merck Manual text.
>
> **Interpretation:** A nine-step protocol is returned (early recognition, fluid resuscitation, antibiotics within the first hour, source control, glucose control, monitoring, organ-dysfunction management). This is a large improvement over the one-line baseline. The source shown is the PDF path only.

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [59]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

response_with_rag_2 = generate_rag_response(
    user_input=question_2,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

print(response_with_rag_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =     237.63 ms /   336 runs   (    0.71 ms per token,  1413.99 tokens per second)
llama_print_timings: prompt eval time =    8041.13 ms /  2643 tokens (    3.04 ms per token,   328.68 tokens per second)
llama_print_timings:        eval time =   31866.53 ms /   335 runs   (   95.12 ms per token,    10.51 tokens per second)
llama_print_timings:       total time =   41992.16 ms /  2978 tokens


The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). However, these classic findings appear in < 50% of patients.

Unfortunately, appendicitis cannot be cured via medicine; surgical intervention is necessary. The recommended surgical procedure is appendectomy, which involves removing the inflamed appendix. The surgery can be performed through laparoscopy or open surgery, depending on the severity of the condition and the patient's overall health. In some cases, the appendix may be removed through a minimally invasive procedure called a laparoscopic appendectomy, which involves making several small incisions in 

> **Objective:** Answer the three-part appendicitis question with RAG.
>
> **Interpretation:** All three parts are covered: symptoms (periumbilical pain shifting to the right lower quadrant, McBurney's point tenderness, classic findings in under half of patients), that medicine alone does not cure it, and appendectomy by laparoscopic or open surgery. It also warns that delay risks perforation and peritonitis.

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [60]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

response_with_rag_3 = generate_rag_response(
    user_input=question_3,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

print(response_with_rag_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      89.73 ms /   137 runs   (    0.65 ms per token,  1526.79 tokens per second)
llama_print_timings: prompt eval time =    8427.57 ms /  2970 tokens (    2.84 ms per token,   352.41 tokens per second)
llama_print_timings:        eval time =   13463.36 ms /   136 runs   (   99.00 ms per token,    10.10 tokens per second)
llama_print_timings:       total time =   22659.42 ms /  3106 tokens


The sudden patchy hair loss you're describing is commonly known as alopecia areata. The possible causes include autoimmune disorders, allergies, fungal infections, or physical stress. Treatment options may include corticosteroids, minoxidil, anthralin, or immunotherapy. It's essential to consult a dermatologist for a proper diagnosis and personalized treatment plan.

Source: The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 86. Hair Disorders, pp. 848-850.


> **Objective:** Answer the patchy hair loss question with RAG.
>
> **Interpretation:** The answer identifies alopecia areata, lists causes and treatments (corticosteroids, minoxidil, anthralin, immunotherapy) and advises seeing a dermatologist. Its chapter and page citation was not in the supplied metadata, so verify it before use.

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [61]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

response_with_rag_4 = generate_rag_response(
    user_input=question_4,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

print(response_with_rag_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =     245.99 ms /   364 runs   (    0.68 ms per token,  1479.73 tokens per second)
llama_print_timings: prompt eval time =    8503.40 ms /  2905 tokens (    2.93 ms per token,   341.63 tokens per second)
llama_print_timings:        eval time =   36927.11 ms /   363 runs   (  101.73 ms per token,     9.83 tokens per second)
llama_print_timings:       total time =   47478.15 ms /  3268 tokens


The treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, may include:

1. Early intervention by rehabilitation specialists to establish baseline findings and prevent secondary disabilities.
2. Cognitive therapy, which may be begun immediately after injury and continued for months or years, to address memory loss, problem-solving skills, language, and other cognitive deficits.
3. Physical therapy to improve mobility and prevent contractures, as well as occupational therapy to redevelop fine motor skills.
4. Bladder and bowel management programs to teach toileting techniques and manage autonomic dysreflexia and neurogenic pain.
5. Vocational rehabilitation to assess cognitive capabilities and identify possible work sites, as well as recreational therapy to facilitate participation in hobbies and other activities.
6. Emotional care to combat depersonalization and depression, and educate t

> **Objective:** Answer the brain-injury treatment question with RAG.
>
> **Interpretation:** A rehabilitation-focused list is returned: early rehabilitation, cognitive therapy, physical and occupational therapy, bladder and bowel programmes, vocational and recreational therapy, emotional care, and surgery for intracranial haematoma. It is far more specific than the baseline.

In [62]:
# Create the DataFrame
RAG_result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "RAG_responses": [response_with_rag_1, response_with_rag_2, response_with_rag_3, response_with_rag_4]
})

# Display the DataFrame
RAG_result_df.head()

,questions,RAG_responses
0,What is the protocol for managing sepsis in a ...,"According to the Merck Manual, the protocol fo..."
1,"What are the common symptoms for appendicitis,...",The common symptoms for appendicitis include e...
2,What are the effective treatments or solutions...,The sudden patchy hair loss you're describing ...
3,What treatments are recommended for a person w...,The treatments recommended for a person who ha...


> **Objective:** Collect the four RAG answers in a DataFrame.
>
> **Interpretation:** Compared with the baseline table, answers are longer, structured and manual-based. Speed cost: about 22-52 seconds per answer versus 2-6 seconds.

## Output Evaluation

In [63]:
# LLM-as-judge rubric 1: groundedness (is the answer supported by the context?)
medical_groundedness_rater_system_message = """
You are an impartial evaluator of AI-generated answers to medical questions.

You will receive a question (###Question), the context used by the AI (###Context) and the AI's answer (###Answer).

Task: rate how well the answer is GROUNDED in the context, i.e. uses only information that is present in the context.

Scale:
1 - Not grounded at all
2 - Grounded only to a limited extent
3 - Grounded to a good extent
4 - Mostly grounded
5 - Completely grounded

Method: check every claim in the answer against the context and penalize any claim the context does not support.

Output: reply with ONLY the final score as a single digit from 1 to 5. No explanation.
"""

In [64]:
# LLM-as-judge rubric 2: relevance (does the answer address the question?)
medical_relevance_rater_system_message = """
You are an impartial evaluator of AI-generated answers to medical questions.

You will receive a question (###Question), the context used by the AI (###Context) and the AI's answer (###Answer).

Task: rate how well the answer addresses ALL important parts of the question, based on the context.

Scale:
1 - Not relevant at all
2 - Relevant only to a limited extent
3 - Relevant to a good extent
4 - Mostly relevant
5 - Completely relevant

Method: list the parts of the question, then check whether the answer covers each one.

Output: reply with ONLY the final score as a single digit from 1 to 5. No explanation.
"""

In [65]:
# Judge user prompt template
medical_rater_user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

> **Objective:** Define the two LLM-as-judge rubrics (groundedness and relevance, 1-5) and the template that supplies question, context and answer.
>
> **Interpretation:** Groundedness checks that the answer stays inside the retrieved context. Relevance checks that it covers every part of the question. The cleaned rubrics ask for a single digit, which makes score extraction reliable.

In [66]:
def generate_ground_relevance_response(user_input, response, retriever,
                                       groundedness_system_message,
                                       relevance_system_message,
                                       user_message_template,
                                       k=5, max_tokens=500, temperature=0, top_p=0.95):

    # Retrieve context and combine into a string
    relevant_chunks = retriever.invoke(user_input)
    if not relevant_chunks:
        return "No context found.", "No context found."

    context = "\n\n".join(
        [f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}" for doc in relevant_chunks]
    )

    # Fill user message template
    user_message = user_message_template.format(
        question=user_input,
        context=context,
        answer=response
    )

    # Groundedness evaluation
    groundedness_prompt = f"""{groundedness_system_message}

{user_message}

Score:
"""
    groundedness_response = lcpp_llm(
        prompt=groundedness_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        echo=False
    )
    groundedness_result = groundedness_response["choices"][0]["text"].strip()

    # Relevance evaluation
    relevance_prompt = f"""{relevance_system_message}

{user_message}

Score:
"""
    relevance_response = lcpp_llm(
        prompt=relevance_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        echo=False
    )
    relevance_result = relevance_response["choices"][0]["text"].strip()

    # Return the textual responses
    return groundedness_result, relevance_result

> **Objective:** Retrieve the context again for a question, then ask the LLM to score the given answer for groundedness and relevance.
>
> **Interpretation:** The same 13B model that writes the answers also grades them, which favours lenient scoring. Judge prompts reached about 3,200-3,900 tokens, close to the 4,096 limit.

#### **Evaluation 1: Prompt Engineering Response Evaluation**

In [67]:
llm_judge_prompt_ground_1, llm_judge_prompt_rel_1 = generate_ground_relevance_response(
    user_input=question_1,
    response=llm_response_1,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

# Print the results
print(llm_judge_prompt_ground_1, end="\n\n")
print(llm_judge_prompt_rel_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      80.77 ms /   140 runs   (    0.58 ms per token,  1733.27 tokens per second)
llama_print_timings: prompt eval time =   10902.13 ms /  3604 tokens (    3.03 ms per token,   330.58 tokens per second)
llama_print_timings:        eval time =   14397.12 ms /   139 runs   (  103.58 ms per token,     9.65 tokens per second)
llama_print_timings:       total time =   26037.00 ms /  3743 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      59.88 ms /   107 runs   (    0.56 ms per token,  1786.82 tokens per second)
llama_print_timings: prompt eval time =   10060.40 ms /  3532 tokens (    2.85 ms per token,   351.08 tokens per second)
llama_print_timings:        eval time =   10813.26 ms /   106 runs   (  102.01 ms per token,     9.80 tokens per second)
llama_print_timings:       to

5 - The answer is completely grounded in the context.

The answer provides a detailed explanation of the Surviving Sepsis Campaign (SSC) guidelines for managing sepsis in a critical care unit, which is entirely based on the provided medical context. The answer covers all the essential aspects of sepsis management, including fluid resuscitation, antibiotics, and monitoring for organ dysfunction. The answer also references specific pages from the provided medical manual, demonstrating a thorough understanding of the context and its application to the question. Therefore, the answer is completely grounded in the context, and I assign a score of 5.

5 - The answer is completely relevant. The Surviving Sepsis Campaign (SSC) guidelines provide a comprehensive protocol for managing sepsis in a critical care unit, including early recognition, prompt administration of antibiotics and vasopressors, and careful monitoring of vital signs and laboratory values. The answer fully addresses all aspect

In [68]:
llm_judge_prompt_ground_2, llm_judge_prompt_rel_2 = generate_ground_relevance_response(
    user_input=question_2,
    response=llm_response_2,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

print(llm_judge_prompt_ground_2, end="\n\n")
print(llm_judge_prompt_rel_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      41.51 ms /    67 runs   (    0.62 ms per token,  1614.07 tokens per second)
llama_print_timings: prompt eval time =    8211.53 ms /  2940 tokens (    2.79 ms per token,   358.03 tokens per second)
llama_print_timings:        eval time =    6309.65 ms /    66 runs   (   95.60 ms per token,    10.46 tokens per second)
llama_print_timings:       total time =   14900.25 ms /  3006 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =       1.33 ms /     2 runs   (    0.66 ms per token,  1504.89 tokens per second)
llama_print_timings: prompt eval time =    8163.45 ms /  2917 tokens (    2.80 ms per token,   357.32 tokens per second)
llama_print_timings:        eval time =      89.51 ms /     1 runs   (   89.51 ms per token,    11.17 tokens per second)
llama_print_timings:       to

The answer is completely grounded in the provided medical context. The answer provides a detailed list of common symptoms of appendicitis and their descriptions, which is entirely based on the provided medical context. Therefore, the groundedness score is 5, indicating that the answer is completely grounded in the provided medical context.

5


In [69]:
llm_judge_prompt_ground_3, llm_judge_prompt_rel_3 = generate_ground_relevance_response(
    user_input=question_3,
    response=llm_response_3,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

print(llm_judge_prompt_ground_3, end="\n\n")
print(llm_judge_prompt_rel_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      17.06 ms /    28 runs   (    0.61 ms per token,  1641.07 tokens per second)
llama_print_timings: prompt eval time =    9372.24 ms /  3279 tokens (    2.86 ms per token,   349.86 tokens per second)
llama_print_timings:        eval time =    2691.23 ms /    27 runs   (   99.68 ms per token,    10.03 tokens per second)
llama_print_timings:       total time =   12221.12 ms /  3306 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      14.73 ms /    26 runs   (    0.57 ms per token,  1764.63 tokens per second)
llama_print_timings: prompt eval time =    9318.06 ms /  3256 tokens (    2.86 ms per token,   349.43 tokens per second)
llama_print_timings:        eval time =    2465.77 ms /    25 runs   (   98.63 ms per token,    10.14 tokens per second)
llama_print_timings:       to

4/5

Please provide the actual text you would like me to read and I'll be happy to assist you further.

4

Please provide the actual text you would like me to read and I'll be happy to assist you further.


In [70]:
llm_judge_prompt_ground_4, llm_judge_prompt_rel_4 = generate_ground_relevance_response(
    user_input=question_4,
    response=llm_response_4,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

print(llm_judge_prompt_ground_4, end="\n\n")
print(llm_judge_prompt_rel_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      16.06 ms /    28 runs   (    0.57 ms per token,  1743.35 tokens per second)
llama_print_timings: prompt eval time =    9266.33 ms /  3245 tokens (    2.86 ms per token,   350.19 tokens per second)
llama_print_timings:        eval time =    2648.50 ms /    27 runs   (   98.09 ms per token,    10.19 tokens per second)
llama_print_timings:       total time =   12068.26 ms /  3272 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =       1.05 ms /     2 runs   (    0.53 ms per token,  1899.34 tokens per second)
llama_print_timings: prompt eval time =    9223.69 ms /  3222 tokens (    2.86 ms per token,   349.32 tokens per second)
llama_print_timings:        eval time =      96.37 ms /     1 runs   (   96.37 ms per token,    10.38 tokens per second)
llama_print_timings:       to

5/5

Please provide the actual text you would like me to read and I'll be happy to assist you further!

5


In [71]:
import re

# Helper function to extract the numeric score
def extract_score(score_text):
    match = re.search(r'(\d+)', score_text)
    return int(match.group(1)) if match else None

# Create a DataFrame to store the base prompt evaluation results
prompt_evaluation_df = pd.DataFrame({
    "question": [question_1, question_2, question_3, question_4],
    "base_prompt_response": [llm_response_1, llm_response_2, llm_response_3, llm_response_4],
    "groundedness_score": [
        extract_score(llm_judge_prompt_ground_1),
        extract_score(llm_judge_prompt_ground_2),
        extract_score(llm_judge_prompt_ground_3),
        extract_score(llm_judge_prompt_ground_4)
    ],
    "relevance_score": [
        extract_score(llm_judge_prompt_rel_1),
        extract_score(llm_judge_prompt_rel_2),
        extract_score(llm_judge_prompt_rel_3),
        extract_score(llm_judge_prompt_rel_4)
    ]
})

prompt_evaluation_df['groundedness_score'] = pd.to_numeric(prompt_evaluation_df['groundedness_score'], errors='coerce')
prompt_evaluation_df['relevance_score'] = pd.to_numeric(prompt_evaluation_df['relevance_score'], errors='coerce')

# Display the DataFrame
display(prompt_evaluation_df)

,question,base_prompt_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,The Surviving Sepsis Campaign (SSC) guidelines...,5,5
1,"What are the common symptoms for appendicitis,...",Appendicitis is a medical emergency that requi...,5,5
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, also known as alopeci...",4,4
3,What treatments are recommended for a person w...,Treatment options for a person who has sustain...,5,5


> **Objective:** Extract the first number from each judge reply and put all baseline scores in a DataFrame.
>
> **Interpretation:** Taking the first number works for replies like "4/5" but is fragile when a reply begins with other digits. Baseline scores: groundedness 5, 5, 4, 5 and relevance 5, 5, 4, 5.

#### **Observations:**

* Judge scores for the baseline: groundedness 5, 5, 4, 5 (mean 4.75) and relevance 5, 5, 4, 5 (mean 4.75).
* **These scores are not trustworthy.** The baseline answers are only one opening sentence, yet the judge described them as covering "fluid resuscitation, antibiotics and monitoring" (Q1). It scored content that was never generated.
* For Q3 and Q4 the judge did not follow the format: it returned "4/5" or "5/5" followed by "Please provide the actual text you would like me to read", which shows the 13B model struggles to act as a strict evaluator.
* Conclusion: a high score here means the judge was lenient, not that the baseline was good.

#### **Evaluation 2: RAG Response Evaluation**

In [72]:
RAG_ground_1, RAG_rel_1 = generate_ground_relevance_response(
    user_input=question_1,
    response=response_with_rag_1,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

# Print the results
print(RAG_ground_1, end="\n\n")
print(RAG_rel_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      14.62 ms /    24 runs   (    0.61 ms per token,  1641.03 tokens per second)
llama_print_timings: prompt eval time =   11324.57 ms /  3881 tokens (    2.92 ms per token,   342.71 tokens per second)
llama_print_timings:        eval time =    2366.62 ms /    23 runs   (  102.90 ms per token,     9.72 tokens per second)
llama_print_timings:       total time =   13853.26 ms /  3904 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      13.28 ms /    23 runs   (    0.58 ms per token,  1732.19 tokens per second)
llama_print_timings: prompt eval time =   11293.83 ms /  3858 tokens (    2.93 ms per token,   341.60 tokens per second)
llama_print_timings:        eval time =    2252.79 ms /    22 runs   (  102.40 ms per token,     9.77 tokens per second)
llama_print_timings:       to

The answer is completely grounded in the provided medical context, and therefore scores a 5 out of 5.

The answer is completely relevant, addressing all important aspects of the medical question. The score is 5.


In [73]:
RAG_ground_2, RAG_rel_2 = generate_ground_relevance_response(
    user_input=question_2,
    response=response_with_rag_2,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

# Print the results
print(RAG_ground_2, end="\n\n")
print(RAG_rel_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      27.66 ms /    40 runs   (    0.69 ms per token,  1446.24 tokens per second)
llama_print_timings: prompt eval time =    9290.91 ms /  3252 tokens (    2.86 ms per token,   350.02 tokens per second)
llama_print_timings:        eval time =    3720.32 ms /    39 runs   (   95.39 ms per token,    10.48 tokens per second)
llama_print_timings:       total time =   13299.30 ms /  3291 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      18.91 ms /    30 runs   (    0.63 ms per token,  1586.88 tokens per second)
llama_print_timings: prompt eval time =    9307.00 ms /  3229 tokens (    2.88 ms per token,   346.94 tokens per second)
llama_print_timings:        eval time =    2772.60 ms /    29 runs   (   95.61 ms per token,    10.46 tokens per second)
llama_print_timings:       to

The answer is mostly grounded in the context provided, but could benefit from more specific details and examples to fully demonstrate its grounding. Therefore, the score is 4 out of 5.

4/5

Please provide any additional information or clarification you would like me to consider when evaluating the relevance of the answer.


In [74]:
RAG_ground_3, RAG_rel_3 = generate_ground_relevance_response(
    user_input=question_3,
    response=response_with_rag_3,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

# Print the results
print(RAG_ground_3, end="\n\n")
print(RAG_rel_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      28.57 ms /    45 runs   (    0.63 ms per token,  1575.19 tokens per second)
llama_print_timings: prompt eval time =    9694.57 ms /  3380 tokens (    2.87 ms per token,   348.65 tokens per second)
llama_print_timings:        eval time =    4316.99 ms /    44 runs   (   98.11 ms per token,    10.19 tokens per second)
llama_print_timings:       total time =   14303.20 ms /  3424 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      24.54 ms /    42 runs   (    0.58 ms per token,  1711.28 tokens per second)
llama_print_timings: prompt eval time =    9669.68 ms /  3357 tokens (    2.88 ms per token,   347.17 tokens per second)
llama_print_timings:        eval time =    4082.46 ms /    41 runs   (   99.57 ms per token,    10.04 tokens per second)
llama_print_timings:       to

The answer is completely grounded in the context provided, as it directly quotes and references specific sections of the provided medical textbook. Therefore, the score is 5, indicating a perfect grounding in the provided context.

4/5

Please provide the actual text of the Merck Manual you're using, as well as any other relevant information you'd like me to consider when evaluating the answer.


In [75]:
RAG_ground_4, RAG_rel_4 = generate_ground_relevance_response(
    user_input=question_4,
    response=response_with_rag_4,
    retriever=retriever,
    groundedness_system_message=medical_groundedness_rater_system_message,
    relevance_system_message=medical_relevance_rater_system_message,
    user_message_template=medical_rater_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0,
    top_p=0.95
)

# Print the results
print(RAG_ground_4, end="\n\n")
print(RAG_rel_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      56.75 ms /    91 runs   (    0.62 ms per token,  1603.44 tokens per second)
llama_print_timings: prompt eval time =   10029.94 ms /  3542 tokens (    2.83 ms per token,   353.14 tokens per second)
llama_print_timings:        eval time =    9113.20 ms /    90 runs   (  101.26 ms per token,     9.88 tokens per second)
llama_print_timings:       total time =   19679.09 ms /  3632 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     719.96 ms
llama_print_timings:      sample time =      14.40 ms /    28 runs   (    0.51 ms per token,  1944.04 tokens per second)
llama_print_timings: prompt eval time =   10035.64 ms /  3519 tokens (    2.85 ms per token,   350.65 tokens per second)
llama_print_timings:        eval time =    2785.03 ms /    27 runs   (  103.15 ms per token,     9.69 tokens per second)
llama_print_timings:       to

The answer is completely grounded in the provided medical context. The answer provides a comprehensive list of treatments that are recommended for a person who has sustained a physical injury to brain tissue, and each treatment is supported by references to the provided medical context. The answer is well-organized and easy to follow, and it demonstrates a clear understanding of the medical information provided. Therefore, the score is 5, completely grounded.

4/5

Please provide the actual text you would like me to read and I'll be happy to assist you further.


In [76]:
# Create a DataFrame to store the RAG evaluation results
RAG_evaluation_df = pd.DataFrame({
    "question": [question_1, question_2, question_3, question_4],
    "RAG_response": [response_with_rag_1, response_with_rag_2, response_with_rag_3, response_with_rag_4],
    "groundedness_score": [
        extract_score(RAG_ground_1),
        extract_score(RAG_ground_2),
        extract_score(RAG_ground_3),
        extract_score(RAG_ground_4)
    ],
    "relevance_score": [
        extract_score(RAG_rel_1),
        extract_score(RAG_rel_2),
        extract_score(RAG_rel_3),
        extract_score(RAG_rel_4)
    ]
})

RAG_evaluation_df['groundedness_score'] = pd.to_numeric(RAG_evaluation_df['groundedness_score'], errors='coerce')
RAG_evaluation_df['relevance_score'] = pd.to_numeric(RAG_evaluation_df['relevance_score'], errors='coerce')

# Display the DataFrame
display(RAG_evaluation_df)

,question,RAG_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,"According to the Merck Manual, the protocol fo...",5,5
1,"What are the common symptoms for appendicitis,...",The common symptoms for appendicitis include e...,4,4
2,What are the effective treatments or solutions...,The sudden patchy hair loss you're describing ...,5,4
3,What treatments are recommended for a person w...,The treatments recommended for a person who ha...,5,4


> **Objective:** Extract the RAG judge scores and show them in a DataFrame.
>
> **Interpretation:** RAG scores: groundedness 5, 4, 5, 5 (mean 4.75) and relevance 5, 4, 4, 4 (mean 4.25). Use them as a rough guide only; see the observations above.

#### **Observations:**

* Judge scores for RAG: groundedness 5, 4, 5, 5 (mean 4.75) and relevance 5, 4, 4, 4 (mean 4.25).
* Read by hand, the RAG answers are clearly **better than the baseline**: they are complete, structured (steps and lists), use Merck Manual wording and address every part of multi-part questions such as Q2.
* The automatic scores do **not** reflect that gap (baseline relevance 4.75 vs RAG 4.25). The reasons are the truncated baseline answers, a lenient and inconsistent judge, and the fact that the same 13B model both generates and grades.
* The Q2 groundedness score of 4 came with a comment about "more specific details", which is a style remark and not a grounding problem.
* Citations show only the PDF file path. The Q3 citation ("Chapter 86, pp. 848-850") was not supplied by the retrieved metadata, so it must be checked against the manual before it is trusted.

## Actionable Insights and Business Recommendations

#### **Actionable insights**

* **RAG turns a general chat model into a manual-grounded assistant.** All four clinical questions were answered with step-by-step content taken from the Merck Manual, which reduces information overload for clinicians.
* **Grounding is the main quality gain.** The baseline knows the topic but gives no protocol and no source. RAG gives usable answers with a traceable origin.
* **Speed trade-off.** Baseline answers took about 2-6 s. RAG answers took about 22-52 s because each prompt carried roughly 2,600-3,600 tokens of retrieved text.
* **The context window is almost full.** Five 512-token chunks plus the system prompt use most of the 4,096-token limit, so there is little room for longer answers or more chunks.
* **Evaluation is the weakest part.** A 13B model grading its own outputs gave inflated and inconsistent scores. Only manual review showed the true difference.

#### **Recommendations**

* **Fix the baseline first:** remove `"\n"` from the `stop` list in `prompt_engineering_response` so the comparison is fair, then re-run the evaluation.
* **Add real citations:** include `d.metadata["page"]` in the context string so every answer can name the manual page.
* **Use a stronger evaluation:** grade with a larger model or a framework such as RAGAS, add gold reference answers for the four questions, and have a clinician spot-check the answers.
* **Improve retrieval:** try a re-ranker, hybrid (BM25 + vector) search, MMR, and section-level metadata filters over the 23 manual sections.
* **Improve speed and capacity:** test a smaller or faster model, cache repeated queries, and use a model with a longer context window.
* **Keep safety controls:** retain the "out of my knowledge base" refusal, add a visible disclaimer, and keep a human in the loop. The tool supports clinicians; it does not replace clinical judgement.
* **Make it usable:** wrap the pipeline in a Streamlit or Gradio app so non-programmers can try it.

<font size=6 color='#4682B4'>Power Ahead</font>
___